In [0]:
%sql
SELECT * FROM DEMOWS.cust_medallion_sch.customers;



customer_id,customer_name,email,city,created_date
1,Kumar,kumar@gmail.com,Chennai,2024-01-01
2,Arun,arun@gmail.com,Bangalore,2024-02-10
3,Priya,priya@gmail.com,Hyderabad,2024-03-15


In [0]:
%sql
SELECT * FROM DEMOWS.cust_medallion_sch.orders;

order_id,customer_id,order_date,amount,status
101,1,2024-04-01,5000.00,Completed
102,1,2024-04-10,2500.00,Pending
103,2,2024-04-11,7000.00,Completed
104,3,2024-04-15,3000.00,Completed


Filter Orders (Completed only)

In [0]:
%sql


SELECT * 
FROM DEMOWS.cust_medallion_sch.orders
WHERE status = 'Completed';

order_id,customer_id,order_date,amount,status
101,1,2024-04-01,5000.00,Completed
103,2,2024-04-11,7000.00,Completed
104,3,2024-04-15,3000.00,Completed


In [0]:
%sql
SELECT 
    c.customer_id,
    c.customer_name,
    o.order_id,
    o.amount,
    o.status
FROM DEMOWS.cust_medallion_sch.customers c
JOIN DEMOWS.cust_medallion_sch.orders o
ON c.customer_id = o.customer_id;

customer_id,customer_name,order_id,amount,status
1,Kumar,101,5000.00,Completed
1,Kumar,102,2500.00,Pending
2,Arun,103,7000.00,Completed
3,Priya,104,3000.00,Completed


Left Join (All customers even without orders)

In [0]:
%sql
SELECT 
    c.customer_id,
    c.customer_name,
    o.order_id,
    o.amount,
    o.status
FROM DEMOWS.cust_medallion_sch.customers c
LEFT JOIN DEMOWS.cust_medallion_sch.orders o
ON c.customer_id = o.customer_id;

customer_id,customer_name,order_id,amount,status
1,Kumar,102,2500.00,Pending
2,Arun,103,7000.00,Completed
3,Priya,104,3000.00,Completed
1,Kumar,101,5000.00,Completed


AGGREGATIONS - Total Sales Per Customer

In [0]:
%sql
SELECT 
    c.customer_name,
    SUM(o.amount) AS total_sales
FROM DEMOWS.cust_medallion_sch.customers c
JOIN DEMOWS.cust_medallion_sch.orders o
ON c.customer_id = o.customer_id
GROUP BY c.customer_name;

customer_name,total_sales
Arun,7000.00
Kumar,7500.00
Priya,3000.00


Top Customer by Revenue

In [0]:
%sql
SELECT 
    customer_id,
    SUM(amount) AS total_sales
FROM DEMOWS.cust_medallion_sch.orders
GROUP BY customer_id
ORDER BY total_sales DESC
LIMIT 1;

customer_id,total_sales
1,7500.00


Running Total per Customer

In [0]:
%sql

SELECT 
    customer_id,
    order_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS running_total
FROM DEMOWS.cust_medallion_sch.orders;




customer_id,order_date,amount,running_total
1,2024-04-01,5000.00,5000.00
1,2024-04-10,2500.00,7500.00
2,2024-04-11,7000.00,7000.00
3,2024-04-15,3000.00,3000.00


Latest Order per Customer

In [0]:
%sql
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY order_date DESC
           ) AS rn
    FROM DEMOWS.cust_medallion_sch.orders
) t
WHERE rn = 1;

order_id,customer_id,order_date,amount,status,rn
102,1,2024-04-10,2500.00,Pending,1
103,2,2024-04-11,7000.00,Completed,1
104,3,2024-04-15,3000.00,Completed,1


MERGE (Upsert) – Delta Feature

In [0]:
%sql
MERGE INTO DEMOWS.cust_medallion_sch.orders AS target
USING DEMOWS.cust_medallion_sch.orders_updates AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


Time Travel (Delta Feature)

In [0]:
%sql
DESCRIBE HISTORY DEMOWS.cust_medallion_sch.orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-03-03T05:46:09.000Z,70550675085612,syedameersyed084@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2378230177734767),d50e03b8-c066-459c-bd64-e59a29353b60,0303-052635-f14n5k9x-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3888, p25FileSize -> 2455, numDeletionVectorsRemoved -> 1, minFileSize -> 2455, numAddedFiles -> 1, maxFileSize -> 2455, p75FileSize -> 2455, p50FileSize -> 2455, numAddedBytes -> 2455)",null,Databricks-Runtime/18.0.x-photon-scala2.13
2,2026-03-03T05:46:07.000Z,70550675085612,syedameersyed084@gmail.com,MERGE,"Map(predicate -> [""(order_id#14666 = order_id#14676)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2378230177734767),d50e03b8-c066-459c-bd64-e59a29353b60,0303-052635-f14n5k9x-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2270, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4619, materializeSourceTimeMs -> 4, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2337, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2165)",null,Databricks-Runtime/18.0.x-photon-scala2.13
1,2026-03-03T05:32:38.000Z,70550675085612,syedameersyed084@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])","List(, null, null, , null, null)",null,null,null,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 1618)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2026-03-03T05:32:07.000Z,70550675085612,syedameersyed084@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.checkpoint.writeStatsAsStruct"":""true"",""delta.enableRowTracking"":""true"",""delta.checkpoint.writeStatsAsJson"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-b2df4851-411e-4477-b7af-e06ee1853561"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-474bdafe-e7f9-4417-bd89-b85ef3b1fc07""}, statsOnLoad -> false)","List(, null, null, , null, null)",null,null,null,null,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT * FROM DEMOWS.cust_medallion_sch.orders VERSION AS OF 0;

order_id,customer_id,order_date,amount,status
